# Environment Setup - E-Commerce Agent Workshop

This notebook prepares the AWS account and local notebook environment for the workshop.

You will learn how the workshop keeps setup state durable across notebooks, how the sample data is loaded, and how to verify that the resources needed by later agent, evaluation, and observability steps are ready. By the end of this notebook, you should have a working AWS sandbox, sample product data, and a local state file that later notebooks can read.

## Step 1: Resolve Notebook Paths

Different editors start Jupyter kernels from different working directories. This cell finds the workshop root and Section 00 folder explicitly so every later file access is based on stable paths.

After running it, check the printed paths. If they point to the workspace repo, the notebook can find setup scripts, sample data, and shared state files reliably.

In [1]:
from pathlib import Path
import sys


def _find_section_dir():
    start = Path.cwd().resolve()
    for parent in [start, *start.parents]:
        for candidate in (parent, parent / "00-prerequisites"):
            if (candidate / "sample_data" / "orders.json").is_file() and (
                candidate / "setup_infrastructure.py"
            ).is_file():
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate 00-prerequisites. Open this notebook from the workshop repo root "
        "or the 00-prerequisites folder."
    )


SECTION_DIR = _find_section_dir()
if str(SECTION_DIR) not in sys.path:
    sys.path.insert(0, str(SECTION_DIR))

from workshop_paths import read_sample_json, section_file_path

print(f"Workshop section directory: {SECTION_DIR}")

Workshop section directory: /workshop/00-prerequisites


## Step 2: Provision AWS Infrastructure

This cell creates the baseline AWS resources used by the workshop and loads the starter product data.

The important learning point is resource ownership: the setup script writes the identifiers it creates into workshop state, so later notebooks do not need to rediscover or hard-code table names. Read the output for created or reused resources and for any permission errors that need attention before continuing.

In [2]:
# Detect AWS region from environment or use default
import boto3

session = boto3.Session()
AWS_REGION = session.region_name or 'us-west-2'
print(f"Using AWS Region: {AWS_REGION}")

Using AWS Region: us-east-1


In [3]:
# Run infrastructure setup
import subprocess

subprocess.run(
    [sys.executable, str(section_file_path("setup_infrastructure.py", section_dir=SECTION_DIR)), "--region", AWS_REGION],
    check=True,
)

Infrastructure Setup initialized
  Region: us-east-1
  Account: 180294215757
  Prefix: ecommerce-workshop

Starting Infrastructure Setup

1. Creating DynamoDB Tables...
  ✅ ecommerce-workshop-orders: Already exists
  ✅ ecommerce-workshop-accounts: Already exists

2. Loading Sample Data...
  ✅ Loaded 10 orders
  ✅ Loaded 8 accounts

3. Creating Products Table...
  ✅ ecommerce-workshop-products: Already exists

4. Loading Product Data...
  ✅ Loaded 10 products
  ✅ Loaded store policies

5. Creating SSM Parameters...
  ✅ ecommerce-workshop-orders-table: ecommerce-workshop-orders
  ✅ ecommerce-workshop-accounts-table: ecommerce-workshop-accounts
  ✅ ecommerce-workshop-products-table: ecommerce-workshop-products

6. Recording Workshop State Manifest...
  ✅ State manifest: /workshop/00-prerequisites/workshop_state.json

INFRASTRUCTURE SETUP COMPLETE!

Resources created:
  - DynamoDB: ecommerce-workshop-orders
  - DynamoDB: ecommerce-workshop-accounts
  - DynamoDB: ecommerce-workshop-products

CompletedProcess(args=['/workshop/.venv/bin/python', '/workshop/00-prerequisites/setup_infrastructure.py', '--region', 'us-east-1'], returncode=0)

## Step 3: Verify Infrastructure

This cell checks whether the required resources exist and whether optional convenience settings are available.

Treat this as the workshop health check. After running it, inspect the pass/fail summary. Required checks should pass before continuing; optional checks explain which convenience features are available or skipped.


In [4]:
# Verify infrastructure is ready
import subprocess

subprocess.run(
    [sys.executable, str(section_file_path("verify_infrastructure.py", section_dir=SECTION_DIR))],
    check=True,
)


E-Commerce Agent Workshop - Infrastructure Verification

AWS Region: us-east-1

1. Checking AWS Identity...
  ✅ AWS Identity: arn:aws:sts::180294215757:assumed-role/code-editor-CodeEditorInstanceBootstrapRole-MjlrZtuxDVFd/i-0270ac1c56585798e
  ✅ Account: 180294215757

2. Checking DynamoDB Tables...
  ✅ ecommerce-workshop-orders: ACTIVE (10 items)
  ✅ ecommerce-workshop-accounts: ACTIVE (8 items)
  ✅ ecommerce-workshop-products: ACTIVE (11 items)

3. Checking Bedrock Model Access...
  ✅ Model: anthropic.claude-sonnet-4-6
   Note: Workshop uses global inference profile:
   - global.anthropic.claude-sonnet-4-6

4. Checking SSM Parameters...
  ✅ Parameter: ecommerce-workshop-orders-table
  ✅ Parameter: ecommerce-workshop-accounts-table
  ✅ Parameter: ecommerce-workshop-products-table

5. Checking Advanced Readiness for Later Modules...
   These checks do not create AgentCore or evaluation resources.
  ✅ AgentCore Runtime: list_agent_runtimes API is reachable
  ✅ AgentCore Gateway: list_ga

CompletedProcess(args=['/workshop/.venv/bin/python', '/workshop/00-prerequisites/verify_infrastructure.py'], returncode=0)

## Step 4: Inspect The Sample Data

This cell loads the local sample data used to seed the product catalog.

Before building an agent, look at the product categories, product IDs, inventory fields, and account/order examples. These records shape the questions the agent can answer and the evaluation cases you will see later.

In [5]:
import textwrap
import pandas as pd
from IPython.display import display

# Load sample data from local files (no AWS connection needed)
orders_data = read_sample_json("orders.json", section_dir=SECTION_DIR)
accounts_data = read_sample_json("accounts.json", section_dir=SECTION_DIR)
products_data = read_sample_json("products.json", section_dir=SECTION_DIR)

print("Sample data loaded successfully!")
print(f"  - {len(accounts_data['accounts'])} customer accounts")
print(f"  - {len(orders_data['orders'])} orders")
print(f"  - {len(products_data['products'])} products in the catalog")


Sample data loaded successfully!
  - 8 customer accounts
  - 10 orders
  - 10 products in the catalog


In [6]:

# Display customer accounts with membership tier breakdown
accounts_df = pd.DataFrame([
    {
        "customer_id": a["customer_id"],
        "name": f"{a['first_name']} {a['last_name']}",
        "membership_tier": a["membership_tier"],
        "account_status": a["account_status"],
        "total_orders": a["total_orders"],
        "total_spent": f"${a['total_spent']:,.2f}",
        "member_since": a["created_date"],
    }
    for a in accounts_data["accounts"]
])

print("=== Customer Accounts ===")
display(accounts_df)

print("\n--- Membership Tier Distribution ---")
tier_counts = accounts_df["membership_tier"].value_counts().rename("count").to_frame()
display(tier_counts)

print("\n--- Account Status Distribution ---")
status_counts = accounts_df["account_status"].value_counts().rename("count").to_frame()
display(status_counts)


=== Customer Accounts ===


,customer_id,name,membership_tier,account_status,total_orders,total_spent,member_since
0,CUST-1001,John Smith,gold,active,15,"$2,456.78",2022-03-15
1,CUST-1002,Sarah Johnson,platinum,active,42,"$8,934.56",2021-08-22
2,CUST-1003,Mike Wilson,standard,active,3,$549.99,2024-01-10
3,CUST-1004,Emily Davis,gold,active,12,"$1,876.43",2023-05-18
4,CUST-1005,Robert Brown,standard,active,5,$623.45,2024-06-01
5,CUST-1006,Lisa Martinez,standard,suspended,8,"$1,245.67",2023-11-20
6,CUST-1007,David Lee,gold,active,23,"$4,567.89",2022-09-30
7,CUST-1008,Jennifer Taylor,standard,active,2,$399.98,2024-10-05



--- Membership Tier Distribution ---


,count
membership_tier,
standard,4
gold,3
platinum,1



--- Account Status Distribution ---


,count
account_status,
active,7
suspended,1


In [7]:

# Display orders with status breakdown
orders_df = pd.DataFrame([
    {
        "order_id": o["order_id"],
        "customer_id": o["customer_id"],
        "status": o["status"],
        "order_date": o["order_date"],
        "items": len(o["items"]),
        "total": f"${o['total']:,.2f}",
    }
    for o in orders_data["orders"]
])

print("=== Orders ===")
display(orders_df)

print("\n--- Order Status Distribution ---")
print("(These represent the different customer service scenarios agents will handle)")
status_counts = orders_df["status"].value_counts().rename("count").to_frame()
display(status_counts)

print("\n--- Orders per Customer ---")
orders_per_customer = (
    orders_df.groupby("customer_id")
    .agg(num_orders=("order_id", "count"))
    .sort_values("num_orders", ascending=False)
)
display(orders_per_customer)


=== Orders ===


,order_id,customer_id,status,order_date,items,total
0,ORD-2024-10001,CUST-1001,delivered,2024-12-15,1,$79.99
1,ORD-2024-10002,CUST-1002,shipped,2024-12-28,2,$359.97
2,ORD-2024-10003,CUST-1003,processing,2025-01-02,1,$449.99
3,ORD-2024-10004,CUST-1001,delivered,2024-12-01,2,$89.98
4,ORD-2024-10005,CUST-1004,pending,2025-01-05,1,$599.99
5,ORD-2024-10006,CUST-1005,return_requested,2024-12-20,1,$149.99
6,ORD-2024-10007,CUST-1006,refunded,2024-11-25,1,$199.99
7,ORD-2024-10008,CUST-1002,delivered,2024-10-15,2,$129.98
8,ORD-2024-10009,CUST-1007,shipped,2025-01-06,2,$189.98
9,ORD-2024-10010,CUST-1008,cancelled,2024-12-30,1,$199.99



--- Order Status Distribution ---
(These represent the different customer service scenarios agents will handle)


,count
status,
delivered,3
shipped,2
processing,1
pending,1
return_requested,1
refunded,1
cancelled,1



--- Orders per Customer ---


,num_orders
customer_id,
CUST-1001,2
CUST-1002,2
CUST-1003,1
CUST-1004,1
CUST-1005,1
CUST-1006,1
CUST-1007,1
CUST-1008,1


In [8]:

# Display product catalog and store policies
products_df = pd.DataFrame([
    {
        "product_id": p["product_id"],
        "name": p["name"],
        "category": p["category"],
        "price": f"${p['price']:.2f}",
        "in_stock": "Yes" if p["in_stock"] else "No",
        "stock_qty": p["stock_quantity"],
        "rating": p["rating"],
        "warranty": p["warranty"],
    }
    for p in products_data["products"]
])

print("=== Product Catalog ===")
display(products_df)

print("\n--- Products by Category ---")
category_summary = (
    products_df.assign(price_raw=[p["price"] for p in products_data["products"]])
    .groupby("category")
    .agg(count=("product_id", "count"), avg_price=("price_raw", "mean"))
    .rename(columns={"count": "# Products", "avg_price": "Avg Price ($)"})
    .round(2)
)
display(category_summary)

print("\n=== Store Policies ===")
for key, policy_text in products_data["policies"].items():
    print(f"\n{key.replace('_', ' ').title()}:")
    # Word-wrap at ~100 chars
    import textwrap
    for line in textwrap.wrap(policy_text, width=100):
        print(f"  {line}")


=== Product Catalog ===


,product_id,name,category,price,in_stock,stock_qty,rating,warranty
0,PROD-001,Wireless Bluetooth Headphones,Audio,$79.99,Yes,150,4.5,1 year manufacturer warranty
1,PROD-015,Smart Watch Pro,Wearables,$299.99,Yes,75,4.7,2 year manufacturer warranty
2,PROD-016,Watch Band - Leather,Accessories,$29.99,Yes,200,4.3,90 days
3,PROD-042,"4K Ultra HD Monitor 27""",Monitors,$449.99,Yes,45,4.6,3 year manufacturer warranty
4,PROD-008,USB-C Hub 7-in-1,Accessories,$49.99,Yes,300,4.4,1 year
5,PROD-055,Noise Canceling Earbuds,Audio,$149.99,Yes,120,4.5,1 year
6,PROD-101,Ergonomic Office Chair,Furniture,$599.99,Yes,25,4.8,"5 year warranty on frame, 2 years on components"
7,PROD-077,Gaming Mechanical Keyboard RGB,Gaming,$159.99,Yes,60,4.6,2 years
8,PROD-088,Webcam 4K HDR,Cameras,$199.99,No,0,4.7,2 years
9,PROD-033,Portable Bluetooth Speaker,Audio,$199.99,Yes,80,4.4,1 year



--- Products by Category ---


,# Products,Avg Price ($)
category,,
Accessories,2,39.99
Audio,3,143.32
Cameras,1,199.99
Furniture,1,599.99
Gaming,1,159.99
Monitors,1,449.99
Wearables,1,299.99



=== Store Policies ===

General Return Policy:
  Most items can be returned within 30 days of delivery for a full refund. Items must be in original
  condition with all packaging and accessories. Some categories (hygiene products, opened software)
  may have restrictions.

Warranty Claims:
  To file a warranty claim, contact customer service with your order number and description of the
  issue. We may request photos or videos of the defect. Approved claims will receive repair,
  replacement, or refund at our discretion.

Price Match:
  We offer price matching within 14 days of purchase if you find a lower price from an authorized
  retailer. Contact customer service with proof of the lower price.

Shipping Policy:
  Standard shipping (5-7 business days) is free for orders over $50. Express shipping (2-3 business
  days) is $9.99. Overnight shipping is $19.99. Alaska and Hawaii may have additional fees.


## Setup Complete

At this point, you should know three things:

1. Where the workshop root and state files are.
2. Which AWS resources were created or reused.
3. What product data the agent will use.

Continue when the verification output is clean. The next notebook builds the local Product Catalog Agent and tests its role-based behavior.